In [1]:
#kernel thesis clean4
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import math
import torch.nn.functional as F

In [2]:
df = pd.read_pickle("C:\\Users\\Patrick\\Masterthesis\\Benchmarks\\screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [3]:
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]  
y_data = np.array(df['class_values'].tolist())
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)
angle = np.array(df['angle_values'].tolist())[..., np.newaxis]
phase = np.array(df['step_values'].tolist())[..., np.newaxis]

angle = np.transpose(angle, (0, 2, 1))
phase = np.transpose(phase, (0, 2, 1))
x_data = np.transpose(x_data, (0, 2, 1))  #reshaped to (num_samples, num_features, sequence_length)

X_train_full, X_test, angle_train_full, angle_test, phase_train_full, phase_test, y_train_full, y_test = train_test_split(x_data, angle, phase, y_encoded, test_size=0.2, stratify=y_encoded,random_state=42)
X_train, X_val, angle_train, angle_val, phase_train, phase_val, y_train, y_val = train_test_split(X_train_full, angle_train_full, phase_train_full, y_train_full,test_size=0.25,stratify=y_train_full,random_state=42) #0.25 * 0.8 = 0.2

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

#angle und phase tensoren
angle_train = torch.tensor(angle_train, dtype=torch.float32)
angle_val   = torch.tensor(angle_val, dtype=torch.float32)
angle_test  = torch.tensor(angle_test, dtype=torch.float32)

phase_train = torch.tensor(phase_train, dtype=torch.float32)
phase_val   = torch.tensor(phase_val, dtype=torch.float32)
phase_test  = torch.tensor(phase_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, angle_train, phase_train, y_train)
val_dataset   = TensorDataset(X_val, angle_val, phase_val, y_val)
test_dataset  = TensorDataset(X_test, angle_test, phase_test, y_test)



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", X_train.shape, angle_train.shape, phase_train.shape, y_train.shape)
print("Validation:", X_val.shape, angle_val.shape, phase_val.shape, y_val.shape)
print("Test:", X_test.shape, angle_test.shape, phase_test.shape, y_test.shape)

Train: torch.Size([7500, 1, 800]) torch.Size([7500, 1, 800]) torch.Size([7500, 1, 800]) torch.Size([7500])
Validation: torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500])
Test: torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500])


In [3]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [4]:
import torch
import torch.nn as nn
import sys
sys.path.append(r"C:\Users\Patrick\InceptionTime-Pytorch")
from inception import InceptionBlock


class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x.mean(-1)   # safer als view


model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)

## Ausschließlich Hooke Gesetz auf allen phasen



In [34]:
#X_train, angle_train, phase_train, y_train)

# Parameter eingesetzt und auf logits gerechnet:

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Torque_hooke(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("pitch", torch.tensor(0.00146, dtype=torch.float32))
        self.register_buffer("D", torch.tensor(0.004, dtype=torch.float32))
        self.register_buffer("screw_length", torch.tensor(0.0012, dtype=torch.float32))
        #self.mu_class = nn.Parameter(torch.tensor([0.3121, 0.3109, 0.3374, 0.3518, 0.3825, 0.3678, 0.3636, 0.3596], dtype=torch.float32))
        self.mu_class = torch.tensor([1.4645, 1.5339, 1.0691, 0.9956, 1.1142, 1.0758, 1.1309, 1.1107], dtype=torch.float32)
        self.register_buffer("k_phase", torch.tensor([1e1, 1e5, 1e3, 1e5], dtype=torch.float32))
        self.d2 = self.D - 0.6495 * self.pitch

    def forward(self, logits, angle, phase, torque, y_true):

        device = logits.device
        #device = torque.device
        probs = F.softmax(logits, dim=-1)
        mu = torch.sum(probs * self.mu_class.to(device), dim=-1) 
        #mu = self.mu_class[y_true].to(device)  
        angle = angle[:,0,:]      
        torque = torque[:, 0,:]    
        phase = phase[:, 0, 0].long()  
        X_ideal = (self.pitch * angle)/ 360.0
        x = torch.clamp(X_ideal,max=self.screw_length.to(device))
        #x = torch.min(self.screw_length, x_ideal)

        k = self.k_phase.to(device)[phase]  
        k = k.unsqueeze(-1)                 
        force = k * x                     

        mu = mu.unsqueeze(-1)             

        numerator = self.pitch + (torch.pi * mu * self.d2)
        denominator = (torch.pi * self.d2) - (mu * self.pitch)

        torque_pred = force * (self.d2 / 2.0) * (numerator / denominator)
        loss = F.mse_loss(torque_pred, torque)

        return loss

In [50]:
from sklearn.metrics import f1_score
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)
model.to(device)

criterion = nn.CrossEntropyLoss()
physics_loss = Torque_hooke().to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(physics_loss.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=20, min_delta=0.001)

epochs = 50

train_losses = []
train_physics_losses = []
val_losses = []
val_physics_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None
lambda_phys = 1.0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    total_phys = 0.0

    for X_batch, angle_batch, phase_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        angle_batch = angle_batch.to(device)
        phase_batch = phase_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss_class = criterion(outputs, y_batch)
        
        #loss phyisics
        loss_physics = physics_loss(outputs, angle_batch, phase_batch, X_batch, y_batch)
        loss = loss_class + lambda_phys * loss_physics
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_phys += loss_physics.item()

    avg_train_loss = total_loss / len(train_loader)
    avg_train_phys_loss = total_phys / len(train_loader)
    train_losses.append(avg_train_loss)
    train_physics_losses.append(avg_train_phys_loss)
    model.eval()

    val_loss = 0.0
    val_physics_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, angle_val_batch, phase_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            angle_val_batch = angle_val_batch.to(device)
            phase_val_batch = phase_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch)
            loss_class = criterion(outputs, y_val_batch)
            loss_physics = physics_loss(outputs, angle_val_batch, phase_val_batch, X_val_batch, y_val_batch)
            loss = loss_class + lambda_phys * loss_physics

            val_loss += loss.item()
            val_physics_loss += lambda_phys * loss_physics.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    avg_val_phys_loss = val_physics_loss / len(val_loader)
    val_physics_losses.append(avg_val_phys_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Physics Loss: {train_physics_losses[-1]:.4f}, Val Physics Loss: {val_physics_losses[-1]:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/50, Train Loss: 1.9781, Val Loss: 1.9075, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.2314
Epoch 2/50, Train Loss: 1.8261, Val Loss: 1.9540, Train Physics Loss: 0.1621, Val Physics Loss: 0.1642, Val F1 Score: 0.2052
Epoch 3/50, Train Loss: 1.7651, Val Loss: 2.0714, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.1835
Epoch 4/50, Train Loss: 1.7057, Val Loss: 2.5094, Train Physics Loss: 0.1621, Val Physics Loss: 0.1642, Val F1 Score: 0.1808
Epoch 5/50, Train Loss: 1.6655, Val Loss: 4.1101, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.1536
Epoch 6/50, Train Loss: 1.6272, Val Loss: 2.0187, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.2219
Epoch 7/50, Train Loss: 1.5831, Val Loss: 1.7500, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.3112
Epoch 8/50, Train Loss: 1.5710, Val Loss: 1.8287, Train Physics Loss: 0.1623, Val Physics Loss: 0.1642, Val F1 Score: 0.2963


In [51]:
model.load_state_dict(best_model_state)
#torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, phase_test_batch, angle_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        angle_test_batch = angle_test_batch.to(device)
        phase_test_batch = phase_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.0274
Test F1 Macro: 0.5284


In [48]:
physics_loss.mu_class

Parameter containing:
tensor([1.4645, 1.5339, 1.0691, 0.9956, 1.1142, 1.0758, 1.1309, 1.1107],
       device='cuda:0', requires_grad=True)

# 3-cv

In [6]:
import math
from sklearn.metrics import f1_score
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]
y_data = np.array(df['class_values'].tolist())
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

angle = np.array(df['angle_values'].tolist())[..., np.newaxis]
phase = np.array(df['step_values'].tolist())[..., np.newaxis]

angle = np.transpose(angle, (0, 2, 1))
phase = np.transpose(phase, (0, 2, 1))
x_data = np.transpose(x_data, (0, 2, 1))

X_train_full, X_test, angle_train_full, angle_test, phase_train_full, phase_test, y_train_full, y_test = train_test_split(
    x_data, angle, phase, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)
cv_scores = []

train_losses = []
train_physics_losses = []
val_losses = []
val_physics_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None
lambda_phys = 1.0
best_overall_f1 = -np.inf
best_overall_model_state = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]
    angle_train_fold = angle_train_full[train_idx]
    phase_train_fold = phase_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]
    angle_val_fold = angle_train_full[val_idx]
    phase_val_fold = phase_train_full[val_idx]

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train_fold, dtype=torch.float32),
            torch.tensor(angle_train_fold, dtype=torch.float32),
            torch.tensor(phase_train_fold, dtype=torch.float32),
            torch.tensor(y_train_fold, dtype=torch.long)
        ),
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_val_fold, dtype=torch.float32),
            torch.tensor(angle_val_fold, dtype=torch.float32),
            torch.tensor(phase_val_fold, dtype=torch.float32),
            torch.tensor(y_val_fold, dtype=torch.long)
        ),
        batch_size=32,
        shuffle=False
    )

    model = nn.Sequential(
        InceptionBlock(
            in_channels=1,
            n_filters=32,
            kernel_sizes=[5, 11, 23],
            bottleneck_channels=32,
            use_residual=True
        ),
        InceptionBlock(
            in_channels=32 * 4,
            n_filters=32,
            kernel_sizes=[5, 11, 23],
            bottleneck_channels=32,
            use_residual=True
        ),
        nn.AdaptiveAvgPool1d(1),
        Flatten(),
        nn.Linear(32 * 4, 8)
    )

    model.to(device)

    criterion = nn.CrossEntropyLoss()
    physics_loss = Torque_hooke().to(device)
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(physics_loss.parameters()), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    train_losses = []
    train_physics_losses = []
    val_losses = []
    val_physics_losses = []
    val_f1_scores = []
    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        total_loss = 0.0
        total_phys = 0.0

        for X_batch, angle_batch, phase_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            angle_batch = angle_batch.to(device)
            phase_batch = phase_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss_class = criterion(outputs, y_batch)

            loss_physics = physics_loss(outputs, angle_batch, phase_batch, X_batch, y_batch)
            loss = loss_class + lambda_phys * loss_physics
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_phys += loss_physics.item()

        avg_train_loss = total_loss / len(train_loader)
        avg_train_phys_loss = total_phys / len(train_loader)
        train_losses.append(avg_train_loss)
        train_physics_losses.append(avg_train_phys_loss)

        model.eval()

        val_loss = 0.0
        val_physics_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():

            for X_val_batch, angle_val_batch, phase_val_batch, y_val_batch in val_loader:

                X_val_batch = X_val_batch.to(device)
                angle_val_batch = angle_val_batch.to(device)
                phase_val_batch = phase_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch)

                loss_class = criterion(outputs, y_val_batch)
                loss_physics = physics_loss(outputs, angle_val_batch, phase_val_batch, X_val_batch, y_val_batch)

                loss = loss_class + lambda_phys * loss_physics

                val_loss += loss.item()
                val_physics_loss += loss_physics.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        avg_val_phys_loss = val_physics_loss / len(val_loader)
        val_physics_losses.append(avg_val_phys_loss)

        val_f1 = f1_score(all_labels, all_preds, average="macro")
        val_f1_scores.append(val_f1)

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Physics Loss: {train_physics_losses[-1]:.4f}, Val Physics Loss: {val_physics_losses[-1]:.4f}, Val F1 Score: {val_f1:.4f}")

    cv_scores.append(best_val_f1)

    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state

print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

Fold 1
Epoch 1/50, Train Loss: 1.9847, Val Loss: 2.5959, Train Physics Loss: 0.1634, Val Physics Loss: 0.1612, Val F1 Score: 0.1689
Epoch 2/50, Train Loss: 1.8023, Val Loss: 2.6202, Train Physics Loss: 0.1637, Val Physics Loss: 0.1612, Val F1 Score: 0.1503
Epoch 3/50, Train Loss: 1.7407, Val Loss: 1.8500, Train Physics Loss: 0.1636, Val Physics Loss: 0.1612, Val F1 Score: 0.2695
Epoch 4/50, Train Loss: 1.7032, Val Loss: 2.3280, Train Physics Loss: 0.1636, Val Physics Loss: 0.1612, Val F1 Score: 0.1557
Epoch 5/50, Train Loss: 1.6750, Val Loss: 1.9686, Train Physics Loss: 0.1635, Val Physics Loss: 0.1612, Val F1 Score: 0.2175
Epoch 6/50, Train Loss: 1.6619, Val Loss: 2.2485, Train Physics Loss: 0.1636, Val Physics Loss: 0.1612, Val F1 Score: 0.2133
Epoch 7/50, Train Loss: 1.6497, Val Loss: 2.5898, Train Physics Loss: 0.1637, Val Physics Loss: 0.1612, Val F1 Score: 0.2032
Epoch 8/50, Train Loss: 1.5959, Val Loss: 2.0217, Train Physics Loss: 0.1636, Val Physics Loss: 0.1612, Val F1 Score: 

In [7]:
model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)
final_model = model
final_model.to(device)
final_model.load_state_dict(best_overall_model_state)
final_model.eval()

test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),torch.tensor(y_test, dtype=torch.long)),batch_size=32,shuffle=False)

all_preds = []
all_labels = []
test_loss = 0.0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:

        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)
        outputs = final_model(X_test_batch)
        loss = criterion(outputs, y_test_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Final Test F1 Macro: {test_f1:.4f}")

Final Test F1 Macro: 0.4786
